# 🩺 Diabetic Retinopathy Model — Comprehensive Test Metrics

**Objective:** Generate detailed performance metrics for the DR detection model

**Metrics Included:**
- Accuracy, Precision, Recall, F1-Score
- AUC-ROC & AUC-PR
- Confusion Matrix & Classification Report
- Sensitivity, Specificity, PPV, NPV
- Matthews Correlation Coefficient
- ROC & Precision-Recall Curves
- Detailed visualizations

---

## 1️⃣ Install Dependencies

In [1]:
!pip install tensorflow scikit-learn matplotlib seaborn pandas numpy pillow -q

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\Prateek Jain\\Downloads\\CUS00036_Pranjal Tomar_CompleteBackup_09-04-26\\venv\\Lib\\site-packages\\pandas\\tests\\extension\\test_sparse.py'
Check the permissions.


[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2️⃣ Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve, auc, balanced_accuracy_score, 
    matthews_corrcoef, cohen_kappa_score
)

# TensorFlow
from tensorflow.keras.models import load_model

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print('✅ All libraries imported successfully!')

## 3️⃣ Configuration

In [ ]:
# Configuration
IMG_SIZE = (224, 224)
CLASS_NAMES = ['DR (Positive)', 'No_DR (Negative)']
NUM_CLASSES = 2

# Model path - located in parent directory (project root)
import os
notebook_dir = os.path.dirname(os.path.abspath('Model_Evaluation_Metrics.ipynb'))
parent_dir = os.path.dirname(notebook_dir)
MODEL_PATH = os.path.join(parent_dir, 'dr_fedavg_model.h5')

# Test data path - extract dataset.zip or create dataset_test folder with DR/ and No_DR/ subdirectories
TEST_DATA_DIR = os.path.join(parent_dir, 'dataset_test')

print('Configuration set:')
print(f'  IMG_SIZE        : {IMG_SIZE}')
print(f'  CLASS_NAMES     : {CLASS_NAMES}')
print(f'  MODEL_PATH      : {MODEL_PATH}')
print(f'  TEST_DATA_DIR   : {TEST_DATA_DIR}')
print(f'  Model exists    : {os.path.exists(MODEL_PATH)}')
print(f'  Test data exists: {os.path.exists(TEST_DATA_DIR)}')

## 4️⃣ Load Model & Test Data

In [ ]:
# Load model
try:
    model = load_model(MODEL_PATH)
    print(f'✅ Model loaded successfully from: {MODEL_PATH}')
except FileNotFoundError:
    print(f'❌ Model file not found: {MODEL_PATH}')
    print('Please update MODEL_PATH variable')
except Exception as e:
    print(f'❌ Error loading model: {e}')

In [ ]:
# Load test dataset from folder structure
def load_test_dataset(test_data_dir, img_size=(224, 224), max_samples_per_class=100):
    """
    Load test dataset from directory structure:
    test_data_dir/
    ├── DR/
    └── No_DR/
    """
    if not os.path.exists(test_data_dir):
        print(f'❌ Test data directory not found: {test_data_dir}')
        return None, None, None
    
    X_test, y_test, image_names = [], [], []
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
    
    # Load DR images (label=0)
    dr_dir = os.path.join(test_data_dir, 'DR')
    if os.path.exists(dr_dir):
        files = [f for f in os.listdir(dr_dir) if f.lower().endswith(valid_exts)]
        print(f'  Loading DR images: {len(files[:max_samples_per_class])} samples...')
        for fname in files[:max_samples_per_class]:
            try:
                img = Image.open(os.path.join(dr_dir, fname)).convert('RGB')
                img = img.resize(img_size)
                X_test.append(np.array(img, dtype=np.float32) / 255.0)
                y_test.append(0)
                image_names.append(fname)
            except Exception as e:
                print(f'  ⚠️  Failed to load {fname}: {e}')
    
    # Load No_DR images (label=1)
    no_dr_dir = os.path.join(test_data_dir, 'No_DR')
    if os.path.exists(no_dr_dir):
        files = [f for f in os.listdir(no_dr_dir) if f.lower().endswith(valid_exts)]
        print(f'  Loading No_DR images: {len(files[:max_samples_per_class])} samples...')
        for fname in files[:max_samples_per_class]:
            try:
                img = Image.open(os.path.join(no_dr_dir, fname)).convert('RGB')
                img = img.resize(img_size)
                X_test.append(np.array(img, dtype=np.float32) / 255.0)
                y_test.append(1)
                image_names.append(fname)
            except Exception as e:
                print(f'  ⚠️  Failed to load {fname}: {e}')
    
    if len(X_test) == 0:
        print('❌ No images loaded from test directory')
        return None, None, None
    
    X_test = np.array(X_test, dtype=np.float32)
    y_test = np.array(y_test, dtype=np.int32)
    
    print(f'✅ Test dataset loaded: {X_test.shape[0]} images')
    print(f'   - DR samples: {np.sum(y_test == 0)}')
    print(f'   - No_DR samples: {np.sum(y_test == 1)}')
    
    return X_test, y_test, image_names


# Load test data if path is provided
if TEST_DATA_DIR and os.path.exists(TEST_DATA_DIR):
    X_test, y_test, image_names = load_test_dataset(TEST_DATA_DIR)
else:
    print('⚠️  TEST_DATA_DIR not set. Update the path above and run again.')

## 5️⃣ Generate Predictions

In [ ]:
# Generate predictions
print('🔮 Generating predictions on test set...')
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)
y_pred_prob_class1 = y_pred_prob[:, 1]  # Probability of class 1 (No_DR)

print(f'✅ Predictions generated')
print(f'   Prediction shape: {y_pred.shape}')
print(f'   Probability shape: {y_pred_prob.shape}')

## 6️⃣ Calculate All Metrics

In [ ]:
# Calculate metrics
print('📊 Calculating metrics...\n')

# Basic metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
balanced_acc = balanced_accuracy_score(y_test, y_pred)

# AUC metrics
auc_roc = roc_auc_score(y_test, y_pred_prob_class1)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_pred_prob_class1)
auc_pr = auc(recall_curve, precision_curve)

# Correlation metrics
mcc = matthews_corrcoef(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

# Confusion matrix based metrics
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Sensitivity (Recall for positive class)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0

# Specificity (Recall for negative class)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Positive Predictive Value (Precision)
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0

# Negative Predictive Value
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

# False rates
fpr_val = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

# Youden's Index
youdens = sensitivity + specificity - 1

print('✅ All metrics calculated successfully!')

## 7️⃣ Display Detailed Results

In [ ]:
print('\n' + '='*70)
print('   📈 TEST SET RESULTS - COMPREHENSIVE METRICS')
print('='*70)

print('\n📌 BASIC CLASSIFICATION METRICS:')
print(f'  Accuracy              : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'  Precision             : {precision:.4f}')
print(f'  Recall (Sensitivity)  : {recall:.4f}')
print(f'  F1-Score              : {f1:.4f}')
print(f'  Balanced Accuracy     : {balanced_acc:.4f}')

print('\n📊 AUC METRICS:')
print(f'  AUC-ROC               : {auc_roc:.4f}')
print(f'  AUC-PR                : {auc_pr:.4f}')

print('\n🎯 PER-CLASS METRICS:')
print(f'  Sensitivity           : {sensitivity:.4f}  (True Positive Rate)')
print(f'  Specificity           : {specificity:.4f}  (True Negative Rate)')
print(f'  PPV (Precision)       : {ppv:.4f}  (Positive Predictive Value)')
print(f'  NPV                   : {npv:.4f}  (Negative Predictive Value)')
print(f'  Youden\'s Index        : {youdens:.4f}')

print('\n⚠️  ERROR RATES:')
print(f'  False Positive Rate   : {fpr_val:.4f}')
print(f'  False Negative Rate   : {fnr:.4f}')

print('\n🔢 CONFUSION MATRIX BREAKDOWN:')
print(f'  True Positives (TP)   : {tp}')
print(f'  True Negatives (TN)   : {tn}')
print(f'  False Positives (FP)  : {fp}')
print(f'  False Negatives (FN)  : {fn}')

print('\n📐 CORRELATION METRICS:')
print(f'  Matthews Correlation Coefficient : {mcc:.4f}')
print(f'  Cohen\'s Kappa                   : {kappa:.4f}')

print('\n' + '='*70)

## 8️⃣ Classification Report

In [ ]:
print('\n📋 DETAILED CLASSIFICATION REPORT:')
print('='*70)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

## 9️⃣ Confusion Matrix Visualization

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, cbar_kws={'label': 'Count'},
            annot_kws={'fontsize': 14, 'weight': 'bold'})

ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold', pad=20)
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Print interpretation
print('\n📊 Confusion Matrix Interpretation:')
print(f'  Top-Left (TN)  : {tn:3d}  - Correctly identified healthy eyes')
print(f'  Top-Right (FP) : {fp:3d}  - Healthy eyes wrongly classified as DR')
print(f'  Bottom-Left (FN): {fn:3d}  - DR patients missed (false negative)')
print(f'  Bottom-Right (TP): {tp:3d}  - Correctly identified DR')

## 🔟 ROC Curve (AUC-ROC)

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob_class1)

# Plot ROC curve
fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(fpr, tpr, color='#e74c3c', lw=2.5, label=f'ROC Curve (AUC = {auc_roc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.5000)', alpha=0.7)
ax.fill_between(fpr, tpr, alpha=0.15, color='#e74c3c')

ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC Curve - AUC-ROC Metric', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print(f'\n🎯 ROC-AUC Score: {auc_roc:.4f}')
print(f'   Interpretation:')
if auc_roc >= 0.95:
    print(f'   ⭐ Excellent discrimination (>= 0.95)')
elif auc_roc >= 0.90:
    print(f'   ⭐ Very good discrimination (>= 0.90)')
elif auc_roc >= 0.80:
    print(f'   ✅ Good discrimination (>= 0.80)')
elif auc_roc >= 0.70:
    print(f'   ⚠️  Fair discrimination (>= 0.70)')
else:
    print(f'   ❌ Poor discrimination (< 0.70)')

## 1️⃣1️⃣ Precision-Recall Curve (AUC-PR)

In [ ]:
# Plot Precision-Recall curve
fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(recall_curve, precision_curve, color='#3498db', lw=2.5, label=f'PR Curve (AUC = {auc_pr:.4f})')
baseline = np.sum(y_test) / len(y_test)  # Proportion of positive class
ax.axhline(y=baseline, color='gray', linestyle='--', lw=1.5, label=f'Baseline (No_DR prevalence = {baseline:.4f})', alpha=0.7)

ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
ax.set_title('Precision-Recall Curve - AUC-PR Metric', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print(f'\n🎯 AUC-PR Score: {auc_pr:.4f}')
print(f'   Interpretation: Measures precision vs recall trade-off')
print(f'   Higher is better (max = 1.0)')

## 1️⃣2️⃣ Metrics Comparison Bar Chart

In [ ]:
# Key metrics comparison
key_metrics = {
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Sensitivity': sensitivity,
    'Specificity': specificity,
    'AUC-ROC': auc_roc,
    'AUC-PR': auc_pr,
}

fig, ax = plt.subplots(figsize=(14, 6))

metrics_names = list(key_metrics.keys())
metrics_values = list(key_metrics.values())

# Color coding based on performance
colors = ['#2ecc71' if v >= 0.8 else '#f39c12' if v >= 0.6 else '#e74c3c' for v in metrics_values]

bars = ax.bar(metrics_names, metrics_values, color=colors, edgecolor='black', linewidth=2, alpha=0.8)

# Add value labels on bars
for bar, value in zip(bars, metrics_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{value:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_ylim([0, 1.1])
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Key Performance Metrics Comparison', fontsize=14, fontweight='bold', pad=20)
ax.axhline(y=0.8, color='green', linestyle=':', alpha=0.5, label='Good threshold (0.8)')
ax.axhline(y=0.6, color='orange', linestyle=':', alpha=0.5, label='Fair threshold (0.6)')
ax.grid(True, alpha=0.3, axis='y')
ax.legend(loc='lower right')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 1️⃣3️⃣ Comprehensive Metrics Summary Table

In [ ]:
# Create comprehensive metrics dataframe
metrics_data = {
    'Metric': [
        'Accuracy', 'Precision', 'Recall', 'F1-Score', 'Balanced Accuracy',
        'AUC-ROC', 'AUC-PR', 'Matthews Correlation Coefficient', "Cohen's Kappa",
        'Sensitivity', 'Specificity', 'PPV', 'NPV',
        'False Positive Rate', 'False Negative Rate', "Youden's Index",
        'True Positives', 'True Negatives', 'False Positives', 'False Negatives'
    ],
    'Value': [
        f'{accuracy:.4f}', f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{balanced_acc:.4f}',
        f'{auc_roc:.4f}', f'{auc_pr:.4f}', f'{mcc:.4f}', f'{kappa:.4f}',
        f'{sensitivity:.4f}', f'{specificity:.4f}', f'{ppv:.4f}', f'{npv:.4f}',
        f'{fpr_val:.4f}', f'{fnr:.4f}', f'{youdens:.4f}',
        f'{tp}', f'{tn}', f'{fp}', f'{fn}'
    ],
    'Description': [
        'Overall correctness', 'True positives among predicted positives', 'True positives among actual positives',
        'Harmonic mean of precision & recall', 'Average of sensitivity & specificity',
        'Area under ROC curve', 'Area under Precision-Recall curve', 'Correlation coefficient', "Cohen's agreement measure",
        'True positive rate (DR detection)', 'True negative rate (Healthy detection)',
        'Positive predictive value', 'Negative predictive value',
        'False alarm rate', 'Missed detection rate', 'Sensitivity + Specificity - 1',
        'Correctly identified DR cases', 'Correctly identified healthy cases',
        'Healthy wrongly marked as DR', 'DR cases missed'
    ]
}

metrics_df = pd.DataFrame(metrics_data)

print('\n' + '='*100)
print('   COMPREHENSIVE METRICS SUMMARY')
print('='*100)
print(metrics_df.to_string(index=False))
print('='*100)

## 1️⃣4️⃣ Comprehensive Visualization Dashboard

In [ ]:
# Create comprehensive dashboard
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.3)

# 1. Confusion Matrix
ax1 = fig.add_subplot(gs[0, 0])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax1, cbar=False)
ax1.set_title('Confusion Matrix', fontsize=12, fontweight='bold')
ax1.set_ylabel('True Label')
ax1.set_xlabel('Predicted Label')

# 2. ROC Curve
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC = {auc_roc:.4f}')
ax2.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax2.fill_between(fpr, tpr, alpha=0.15, color='#e74c3c')
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1])
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve (AUC-ROC)', fontsize=12, fontweight='bold')
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

# 3. Precision-Recall Curve
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(recall_curve, precision_curve, color='#3498db', lw=2, label=f'AUC-PR = {auc_pr:.4f}')
ax3.set_xlim([0, 1])
ax3.set_ylim([0, 1])
ax3.set_xlabel('Recall')
ax3.set_ylabel('Precision')
ax3.set_title('Precision-Recall Curve (AUC-PR)', fontsize=12, fontweight='bold')
ax3.legend(loc='best')
ax3.grid(True, alpha=0.3)

# 4. Metrics Bar Chart
ax4 = fig.add_subplot(gs[1, 1])
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'AUC-PR']
values_to_plot = [accuracy, precision, recall, f1, auc_roc, auc_pr]
colors_plot = ['#2ecc71' if v >= 0.8 else '#f39c12' if v >= 0.6 else '#e74c3c' for v in values_to_plot]
bars = ax4.bar(metrics_to_plot, values_to_plot, color=colors_plot, edgecolor='black', linewidth=1.5)
for bar, value in zip(bars, values_to_plot):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{value:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax4.set_ylim([0, 1.15])
ax4.set_ylabel('Score')
ax4.set_title('Key Metrics Comparison', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 5. Class metrics
ax5 = fig.add_subplot(gs[2, 0])
class_metrics_names = ['Sensitivity', 'Specificity', 'PPV', 'NPV']
class_metrics_vals = [sensitivity, specificity, ppv, npv]
bars = ax5.bar(class_metrics_names, class_metrics_vals, color='#9b59b6', edgecolor='black', linewidth=1.5, alpha=0.8)
for bar, value in zip(bars, class_metrics_vals):
    height = bar.get_height()
    ax5.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{value:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax5.set_ylim([0, 1.15])
ax5.set_ylabel('Score')
ax5.set_title('Per-Class Metrics', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3, axis='y')
plt.setp(ax5.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 6. Metrics Summary Text Box
ax6 = fig.add_subplot(gs[2, 1])
ax6.axis('off')
summary_text = f'''EVALUATION SUMMARY



Accuracy:       {accuracy:.4f}
Precision:      {precision:.4f}
Recall:         {recall:.4f}
F1-Score:       {f1:.4f}

AUC-ROC:        {auc_roc:.4f}
AUC-PR:         {auc_pr:.4f}

Sensitivity:    {sensitivity:.4f}
Specificity:    {specificity:.4f}

MCC:            {mcc:.4f}
Kappa:          {kappa:.4f}'''

ax6.text(0.1, 0.95, summary_text, transform=ax6.transAxes,
         fontsize=11, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('🩺 COMPREHENSIVE MODEL EVALUATION DASHBOARD', fontsize=16, fontweight='bold', y=0.995)
plt.show()

## 1️⃣5️⃣ Prediction Confidence Distribution

In [ ]:
# Analyze prediction confidence
max_probabilities = np.max(y_pred_prob, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of max probabilities
axes[0].hist(max_probabilities, bins=30, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(x=np.mean(max_probabilities), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(max_probabilities):.3f}')
axes[0].axvline(x=np.median(max_probabilities), color='green', linestyle='--', linewidth=2, label=f'Median = {np.median(max_probabilities):.3f}')
axes[0].set_xlabel('Maximum Prediction Probability', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0].set_title('Distribution of Prediction Confidence', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Boxplot of probabilities by class
dr_probs = max_probabilities[y_test == 0]
no_dr_probs = max_probabilities[y_test == 1]

bp = axes[1].boxplot([dr_probs, no_dr_probs], labels=['DR (True)', 'No_DR (True)'],
                       patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], ['#e74c3c', '#2ecc71']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1].set_ylabel('Maximum Prediction Probability', fontsize=11, fontweight='bold')
axes[1].set_title('Confidence Distribution by True Class', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f'\n📊 Prediction Confidence Analysis:')
print(f'  Mean confidence      : {np.mean(max_probabilities):.4f}')
print(f'  Median confidence    : {np.median(max_probabilities):.4f}')
print(f'  Min confidence       : {np.min(max_probabilities):.4f}')
print(f'  Max confidence       : {np.max(max_probabilities):.4f}')
print(f'  Std confidence       : {np.std(max_probabilities):.4f}')

## 1️⃣6️⃣ Save Results to CSV

In [ ]:
# Save comprehensive metrics to CSV
output_file = 'model_evaluation_metrics.csv'
metrics_df.to_csv(output_file, index=False)

print(f'✅ Metrics saved to: {output_file}')
print(f'\nFirst few rows:')
print(metrics_df.head(10))

## 1️⃣7️⃣ Model Performance Interpretation & Recommendations

In [ ]:
print('\n' + '='*70)
print('   🔍 PERFORMANCE INTERPRETATION & RECOMMENDATIONS')
print('='*70)

print('\n📊 OVERALL PERFORMANCE:')
if accuracy >= 0.95:
    print(f'  ⭐⭐⭐ EXCELLENT - Accuracy {accuracy:.4f} is exceptional')
elif accuracy >= 0.90:
    print(f'  ⭐⭐ VERY GOOD - Accuracy {accuracy:.4f} is excellent')
elif accuracy >= 0.85:
    print(f'  ⭐ GOOD - Accuracy {accuracy:.4f} is good')
elif accuracy >= 0.75:
    print(f'  ⚠️  FAIR - Accuracy {accuracy:.4f} needs improvement')
else:
    print(f'  ❌ POOR - Accuracy {accuracy:.4f} needs significant improvement')

print('\n🎯 CLASS BALANCE ANALYSIS:')
if sensitivity >= 0.95 and specificity >= 0.95:
    print(f'  ✅ Both classes detected well (Sens={sensitivity:.4f}, Spec={specificity:.4f})')
elif sensitivity >= specificity:
    print(f'  ⚠️  Better at detecting DR (Sens={sensitivity:.4f}) than healthy (Spec={specificity:.4f})')
    print(f'     This is good for medical screening (fewer missed cases)')
else:
    print(f'  ⚠️  Better at detecting healthy (Spec={specificity:.4f}) than DR (Sens={sensitivity:.4f})')
    print(f'     This may miss DR cases - consider rebalancing')

print('\n💡 RECOMMENDATIONS:')
if fnr > 0.15:
    print(f'  • High False Negative Rate ({fnr:.4f}) - Consider lowering decision threshold')
if fpr_val > 0.15:
    print(f'  • High False Positive Rate ({fpr_val:.4f}) - Consider raising decision threshold')
if kappa < 0.6:
    print(f'  • Low Cohen\'s Kappa ({kappa:.4f}) - Model agreement could be better')
if mcc < 0.6:
    print(f'  • Low MCC ({mcc:.4f}) - Model performance needs improvement')
else:
    print(f'  ✅ MCC ({mcc:.4f}) indicates good overall performance')

print(f'  • For medical screening, prioritize Sensitivity (recall false negatives)')
print(f'  • Current Sensitivity: {sensitivity:.4f} (detecting DR cases)')
print(f'  • Current Specificity: {specificity:.4f} (detecting healthy cases)')

print('\n' + '='*70)

---

## Summary

This comprehensive evaluation notebook calculates and visualizes:

### ✅ Calculated Metrics:
- **Accuracy, Precision, Recall, F1-Score**
- **AUC-ROC & AUC-PR** (Area Under Curves)
- **Confusion Matrix** (TP, TN, FP, FN)
- **Sensitivity, Specificity, PPV, NPV**
- **Matthews Correlation Coefficient (MCC)**
- **Cohen's Kappa**
- **False Positive & False Negative Rates**
- **Youden's Index**

### 📊 Visualizations Generated:
- Confusion Matrix Heatmap
- ROC Curve with AUC Score
- Precision-Recall Curve
- Metrics Comparison Bar Charts
- Comprehensive Evaluation Dashboard
- Prediction Confidence Distribution

### 💾 Output Files:
- `model_evaluation_metrics.csv` - All metrics in tabular format